In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime

user_activity = pd.read_csv(r"C:\Users\corey\Downloads\da_fitly_user_activity.csv")

customer_support = pd.read_csv(r"C:\Users\corey\Downloads\da_fitly_customer_support.csv")

account_info = pd.read_csv(r"C:\Users\corey\Downloads\da_fitly_account_info.csv")

# Account info data cleaning 

account_info = account_info.drop_duplicates()

account_info['churn_status'] = account_info['churn_status'].replace('Y', 'Churned')

account_info['churn_status'] = account_info['churn_status'].fillna('Not Churned')

account_info['customer_id'] = account_info['customer_id'].str.replace('C', '').astype(int)

account_info = account_info.rename(columns={"customer_id": "user_id"})

account_info['plan_list_price'] = account_info['plan_list_price'].astype(float)

# Customer support data cleaning
customer_support = customer_support.drop_duplicates()

customer_support = customer_support.drop("state", axis=1)

# User activity data cleaning

user_activity = user_activity.drop_duplicates()


table_merge = account_info.merge(customer_support, on='user_id', how='outer')

table_merge2 = table_merge.merge(user_activity, on ='user_id', how='outer')

# 1. Identify all users who submitted ANY support comment
ids_to_delete = (
    table_merge2.loc[
        table_merge2['comments'].notna() & (table_merge2['comments'].str.strip() != ""),
        'user_id'
    ]
    .unique()
)

table_merge2_clean = table_merge2[~table_merge2['user_id'].isin(ids_to_delete)].copy()

churned = table_merge2_clean[table_merge2_clean['churn_status'] == 'Churned']

not_churned = table_merge2_clean[table_merge2_clean['churn_status'] == 'Not Churned']

churned['ticket_time'] = churned['ticket_time'].astype('datetime64[ns]')

not_churned['ticket_time'] = not_churned['ticket_time'].astype('datetime64[ns]')

churned = churned.dropna(subset=['ticket_time'])

churned['hour'] = pd.to_datetime(churned['ticket_time']).dt.hour.astype(int).apply(
    lambda h: pd.to_datetime(h, format='%H').strftime('%I %p').lstrip('0')
)

not_churned['ticket_time'] = pd.to_datetime(not_churned['ticket_time']).dt.hour

churned['event_time'] = pd.to_datetime(churned['event_time']).dt.hour

passive_events = ['read_article', 'watch_video']

passive_only_users = (
    table_merge2_clean
        .groupby('user_id')['event_type']
        .apply(lambda x: set(x).issubset(passive_events))
)

passive_users = table_merge2_clean[
    table_merge2_clean['user_id'].isin(
        passive_only_users[passive_only_users].index
    )
]

passive_users_avg_service = passive_users[passive_users['resolution_time_hours'] <= 7]

order = churned['ticket_time'].value_counts().index

counts = passive_users_avg_service['churn_status'].value_counts(normalize=True)

top6 = churned['hour'].value_counts().nlargest(6).index

palette = ['red'] * 4 + ['lightblue'] * (len(top6) - 4)

sns.countplot(data=churned[churned['hour'].isin(top6)],
              x='hour',
              order=top6,
              palette=palette)
plt.title("Hour of Ticket Submission for Churned Users")
plt.show()